In [1]:
from langchain_ollama import ChatOllama

In [2]:
# OLLAMA_URL = "http://localhost:11434"

In [3]:
OLLAMA_URL = "http://localhost:6006"

In [4]:
selected_model = "llama3.2:latest"

In [5]:
llm = ChatOllama(
    base_url=OLLAMA_URL,
    model=selected_model,
    temperature=0.7
)

In [6]:
llm

ChatOllama(model='llama3.2:latest', temperature=0.7, base_url='http://localhost:6006')

In [7]:
from langchain.schema import (
    Document,
    HumanMessage,
    AIMessage
)

In [8]:
def clean_response(response):
    """清理响应文本，去除不需要的符号和元数据"""
    if isinstance(response, dict):
        content = response.get("answer") or response.get("content") or str(response)
    else:
        content = str(response)
    
    # 清理响应文本
    if "additional_kwargs=" in content:
        content = content.split("additional_kwargs=")[0]  # 移除元数据部分
    
    # 移除 think 标签
    content = content.replace("content='<think>", "")
    # content = content.replace("</think>", "")
    content = content.replace("<think>", "")
    content=content.replace("content='", "")
    content = content.replace("\\n", "\n")  # 处理换行符
    content = content.strip("'\" ")  # 移除首尾的引号和空格
    return content

In [ ]:
# def generate_response(llm, prompt, search_results=None, source_type=None):
#     """生成回答"""
#     try:
#         response = llm.invoke(prompt)
#         if response is None:
#             return "⚠️ AI 没有返回结果，请稍后再试", [], source_type
        
#         # 如果提供了搜索结果，将其转换为原文档格式
#         source_documents = []
        
#         if search_results and 'google' in search_results and 'data' in search_results['google']:
#             for item in search_results['google']['data'][:5]:
#                 title = item.get("title", "")
#                 snippet = item.get("snippet", "")
#                 link = item.get("link", "")
                
#                 # 创建源文档对象
#                 source_documents.append(
#                     Document(
#                         page_content=f"📌 {title}\n📄 摘要：{snippet}\n🔗 来源：{link}\n",
#                         metadata={"source": link, "title": title}
#                     )
#                 )
        
#         return clean_response(response), source_documents, source_type
#     except Exception as e:
#         # logger.error(f"生成响应时出错:  {str(e)}", exc_info=True)
#         return f"⚠️ 发生错误，无法生成回答: {str(e)}", [], source_type

In [9]:
def generate_response(llm, prompt, search_results=None, source_type=None):
    """生成回答"""

    response = llm.invoke(prompt)
    if response is None:
        return "⚠️ AI 没有返回结果，请稍后再试", [], source_type
    
    # 如果提供了搜索结果，将其转换为原文档格式
    source_documents = []
    
    if search_results and 'google' in search_results and 'data' in search_results['google']:
        for item in search_results['google']['data'][:5]:
            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")
            
            # 创建源文档对象
            source_documents.append(
                Document(
                    page_content=f"📌 {title}\n📄 摘要：{snippet}\n🔗 来源：{link}\n",
                    metadata={"source": link, "title": title}
                )
            )
    
    return clean_response(response), source_documents, source_type


In [ ]:
def format_chat_history(chat_history):
    """格式化对话历史"""
    if not chat_history:
        return ""

In [ ]:
formatted_history = format_chat_history(st.session_state.get("chat_history", []))

In [ ]:
user_question = st.session_state.user_question

In [10]:
prompt = f"""
{"None"}

【当前问题】
{"hello"}

📋 【回答要求】：
1. **如果你能100%确定答案，请直接回答**
2. **如果问题涉及最新信息（如“现在”、“今年”、“最近”），请直接说："请查询最新数据"**
3. **如果你不确定答案，请直接说："我不确定，请查询最新信息"**
4. **不要编造信息**
"""

In [18]:
llm.invoke(prompt)

ConnectError: [WinError 10061] 由于目标计算机积极拒绝，无法连接。

In [16]:
generate_response(llm, prompt, None, "direct")[:2]

('⚠️ 发生错误，无法生成回答:  (status code: 502)', [])

In [13]:
answer, source_documents = generate_response(llm, prompt, None, "direct")[:2]

In [14]:
answer

'⚠️ 发生错误，无法生成回答:  (status code: 502)'

In [15]:
source_documents

[]

In [29]:
import requests

In [30]:
session = requests.Session()
session.trust_env = False  # 禁用系统代理

In [32]:
session.get("http://localhost:6006/api/tags")

<Response [200]>

In [37]:
response = session.get("http://localhost:11434/api/tags")

In [38]:
response.status_code

200

In [41]:
response.json()

{'models': [{'name': 'llama2:latest',
   'model': 'llama2:latest',
   'modified_at': '2025-03-08T05:45:58.07384721Z',
   'size': 3826793677,
   'digest': '78e26419b4469263f75331927a00a0284ef6544c1975b826b15abdaef17bb962',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'llama',
    'families': ['llama'],
    'parameter_size': '7B',
    'quantization_level': 'Q4_0'}},
  {'name': 'llama3:latest',
   'model': 'llama3:latest',
   'modified_at': '2025-03-08T05:41:23.077727026Z',
   'size': 4661224676,
   'digest': '365c0bd3c000a25d28ddbf732fe1c6add414de7275464c4e4d1c3b5fcb5d8ad1',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': 'llama',
    'families': ['llama'],
    'parameter_size': '8.0B',
    'quantization_level': 'Q4_0'}},
  {'name': 'llama3.2:latest',
   'model': 'llama3.2:latest',
   'modified_at': '2025-03-01T07:37:59.390488097Z',
   'size': 2019393189,
   'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72',
   '

In [40]:
response.json().get('models', [])

[{'name': 'llama2:latest',
  'model': 'llama2:latest',
  'modified_at': '2025-03-08T05:45:58.07384721Z',
  'size': 3826793677,
  'digest': '78e26419b4469263f75331927a00a0284ef6544c1975b826b15abdaef17bb962',
  'details': {'parent_model': '',
   'format': 'gguf',
   'family': 'llama',
   'families': ['llama'],
   'parameter_size': '7B',
   'quantization_level': 'Q4_0'}},
 {'name': 'llama3:latest',
  'model': 'llama3:latest',
  'modified_at': '2025-03-08T05:41:23.077727026Z',
  'size': 4661224676,
  'digest': '365c0bd3c000a25d28ddbf732fe1c6add414de7275464c4e4d1c3b5fcb5d8ad1',
  'details': {'parent_model': '',
   'format': 'gguf',
   'family': 'llama',
   'families': ['llama'],
   'parameter_size': '8.0B',
   'quantization_level': 'Q4_0'}},
 {'name': 'llama3.2:latest',
  'model': 'llama3.2:latest',
  'modified_at': '2025-03-01T07:37:59.390488097Z',
  'size': 2019393189,
  'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72',
  'details': {'parent_model': '',
   'for

In [18]:
session.get("http://127.0.0.1:11434/api/tags")

<Response [200]>

In [2]:
OLLAMA_URL = "http://127.0.0.1:11434"